# DINO Ablation — Reduced Output Dim (65536 → 4096)

## Ablation Study: Countering Representational Collapse

In our baseline DINO experiments, both **Knee** and **Brain** models collapsed:
- Loss plateaued at **11.0903 ≈ ln(65536)** — the uniform distribution over the output space
- Brain features had **cos sim = 0.997** (complete collapse)
- Knee features had **cos sim = 0.011** (borderline, but loss still flatlined)
- Only **Combined** (2× data, 3 tissue types) trained healthily

### Root Cause
The 65,536-dim output head was overparameterized for ~12,864 training samples.
With only single-anatomy data, there is insufficient distributional diversity for DINO's
self-distillation to learn non-trivial solutions in such a high-dimensional output space.

### Ablation Change
| Parameter | Baseline | Ablation |
|-----------|----------|----------|
| `HEAD_OUT_DIM` | 65,536 | **4,096** |
| Everything else | Same | Same |

This single change reduces the output-to-sample ratio from ~5:1 to ~0.3:1,
making collapse much harder. If successful, it confirms that **DINO's collapse
on low-diversity medical data is driven by output dimensionality, not the
self-distillation objective itself**.

## Time Budget (~12h total)
| Phase | Est. Time |
|-------|-----------|
| Knee (20 epochs + analysis) | ~2h |
| Brain (20 epochs + analysis) | ~2h |
| Combined (20 epochs + analysis) | ~4h |
| **Total** | **~8h** |

In [ ]:
import os
import math
import time
import gc
import json
import numpy as np
import random
import matplotlib.pyplot as plt
import scipy.ndimage

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from tqdm import tqdm
from sklearn.neighbors import NearestNeighbors
from sklearn.manifold import TSNE

# ── Reproducibility ──
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if device == "cuda":
    print(torch.cuda.get_device_name(0))
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Hyperparameters

**All identical to baseline DINO except `HEAD_OUT_DIM = 4096` (was 65536).**

Why 4096?
- Original DINO paper uses 65536 with ImageNet (1.28M images)
- We have ~12,864 images per single-anatomy dataset
- Scaling output dim proportionally: 65536 × (12864/1280000) ≈ 660
- 4096 is a conservative middle ground — still high enough for rich representations,
  low enough to prevent the uniform-distribution collapse
- ln(4096) = 8.318 — if collapse occurs, loss would plateau at ~8.32 (easy to detect)

In [ ]:
# ══════════════════════════════════════════════
# Hyperparameters (shared across all 3 models)
# ══════════════════════════════════════════════
BATCH_SIZE     = 32           # lower due to multi-crop memory
GRAD_ACCUM     = 8            # effective batch = 32 × 8 = 256
EPOCHS         = 20
BASE_LR        = 5e-4         # DINO standard (higher than MAE)
WARMUP_EPOCHS  = 2
MIN_LR         = 1e-6

# Weight decay schedule
WD_START       = 0.04
WD_END         = 0.4

# Teacher EMA schedule
TEACHER_MOMENTUM_START = 0.996
TEACHER_MOMENTUM_END   = 1.0

# DINO temperatures
STUDENT_TEMP   = 0.1
TEACHER_TEMP_START = 0.04
TEACHER_TEMP_END   = 0.07
TEACHER_TEMP_WARMUP = 5      # epochs to warm up teacher temp

# Centering
CENTER_MOMENTUM = 0.9

# ViT-Small encoder (IDENTICAL to baseline)
IMG_SIZE       = 192
LOCAL_CROP_SIZE = 96
PATCH_SIZE     = 16
ENC_DIM        = 384
ENC_DEPTH      = 12
ENC_HEADS      = 6
MLP_RATIO      = 4.0
DROP_PATH      = 0.1

# DINO head — ★ ABLATION CHANGE ★
HEAD_HIDDEN    = 2048
HEAD_BOTTLENECK = 256
HEAD_OUT_DIM   = 4096          # ★ Was 65536 in baseline
HEAD_NLAYERS   = 3

# Multi-crop
N_GLOBAL_CROPS = 2
N_LOCAL_CROPS  = 6

# Evaluation
K_NEIGHBORS    = 5
NUM_WORKERS    = 4
MAX_GRAD_NORM  = 3.0          # DINO uses higher grad clip

NUM_PATCHES_GLOBAL = (IMG_SIZE // PATCH_SIZE) ** 2       # 144
NUM_PATCHES_LOCAL  = (LOCAL_CROP_SIZE // PATCH_SIZE) ** 2  # 36

print("★ ABLATION: HEAD_OUT_DIM = 4096 (baseline was 65536)")
print(f"  → ln(4096) = {np.log(4096):.3f} (collapse plateau if it occurs)")
print(f"  → ln(65536) = {np.log(65536):.3f} (baseline collapse plateau)")
print(f"\nEffective batch size: {BATCH_SIZE * GRAD_ACCUM}")
print(f"Global crops: {N_GLOBAL_CROPS} × {IMG_SIZE}×{IMG_SIZE} ({NUM_PATCHES_GLOBAL} patches)")
print(f"Local crops:  {N_LOCAL_CROPS} × {LOCAL_CROP_SIZE}×{LOCAL_CROP_SIZE} ({NUM_PATCHES_LOCAL} patches)")
print(f"Total crops per image: {N_GLOBAL_CROPS + N_LOCAL_CROPS}")
print(f"Epochs per model: {EPOCHS}")

## Data Pipeline

**Identical to baseline DINO** — same balanced sampling, same multi-crop augmentation.

In [ ]:
# ══════════════════════════════════════════════
# Balanced Sampling + Multi-Crop Augmentations
# ══════════════════════════════════════════════

def get_balanced_paths(root_dir, target_total=None):
    """Deterministic balanced subsampling — IDENTICAL to baseline."""
    groups = {}
    for dirpath, _, filenames in os.walk(root_dir):
        npy_files = sorted(f for f in filenames if f.endswith('.npy'))
        if npy_files:
            folder = os.path.basename(dirpath)
            groups[folder] = [os.path.join(dirpath, f) for f in npy_files]

    if not groups:
        return [], []

    if target_total is None:
        paths, labels = [], []
        for folder in sorted(groups):
            paths.extend(groups[folder])
            labels.extend([folder] * len(groups[folder]))
        return paths, labels

    n_groups = len(groups)
    target_per_group = target_total // n_groups

    paths, labels = [], []
    for folder in sorted(groups):
        group_paths = groups[folder]
        if len(group_paths) <= target_per_group:
            selected = group_paths
        else:
            idx = np.round(np.linspace(0, len(group_paths) - 1, target_per_group)).astype(int)
            selected = [group_paths[i] for i in idx]
        paths.extend(selected)
        labels.extend([folder] * len(selected))
        print(f"    {folder}: {len(selected)}/{len(group_paths)} selected")

    return paths, labels


def _apply_augment(img, do_blur=True):
    """Apply DINO-style augmentations to a 2D numpy array."""
    if random.random() < 0.5:
        img = np.flip(img, axis=1).copy()
    if random.random() < 0.5:
        img = np.flip(img, axis=0).copy()
    if random.random() < 0.5:
        angle = random.uniform(-15, 15)
        img = scipy.ndimage.rotate(img, angle, reshape=False, order=1, mode='reflect')
    # Intensity jitter
    if random.random() < 0.8:
        gain = random.uniform(0.85, 1.15)
        bias = random.uniform(-0.08, 0.08)
        img = img * gain + bias
    # Gaussian blur
    if do_blur and random.random() < 0.5:
        sigma = random.uniform(0.1, 1.5)
        img = scipy.ndimage.gaussian_filter(img, sigma=sigma)
    return np.clip(img, 0.0, 1.0).astype(np.float32)


def _random_crop(img, crop_size):
    """Random crop from 2D image."""
    h, w = img.shape
    top = random.randint(0, h - crop_size)
    left = random.randint(0, w - crop_size)
    return img[top:top+crop_size, left:left+crop_size].copy()


class DINOMultiCropTransform:
    """DINO multi-crop: 2 global + 6 local crops with augmentations."""

    def __init__(self, global_size=192, local_size=96,
                 n_global=2, n_local=6):
        self.global_size = global_size
        self.local_size = local_size
        self.n_global = n_global
        self.n_local = n_local

    def __call__(self, img):
        crops = []
        # Global crops (full resolution with augmentation)
        for i in range(self.n_global):
            aug = _apply_augment(img.copy(), do_blur=(i == 0))
            crops.append(torch.from_numpy(aug).unsqueeze(0))

        # Local crops (random 96×96 patches)
        for _ in range(self.n_local):
            patch = _random_crop(img, self.local_size)
            aug = _apply_augment(patch, do_blur=(random.random() < 0.5))
            crops.append(torch.from_numpy(aug).unsqueeze(0))

        return crops  # list of tensors


class CleanTransform:
    def __call__(self, x):
        return torch.from_numpy(x).unsqueeze(0).float()


class MRIDataset(Dataset):
    def __init__(self, paths, labels, label_map, transform=None, name=""):
        self.paths = paths
        self.label_map = label_map
        self.labels = [label_map[l] for l in labels]
        self.transform = transform or CleanTransform()
        print(f"  [{name}] {len(paths)} slices | Labels: {label_map}")

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = np.load(self.paths[idx]).astype(np.float32)
        return self.transform(img), self.labels[idx]


def dino_collate(batch):
    """Custom collate for multi-crop: returns list of batched crop tensors + labels."""
    crops_list, labels = zip(*batch)
    n_crops = len(crops_list[0])
    batched_crops = []
    for c in range(n_crops):
        batched_crops.append(torch.stack([crops_list[b][c] for b in range(len(batch))]))
    return batched_crops, torch.tensor(labels)

## Model Architecture

**Identical to baseline except DINO Head output: 4096 instead of 65536.**

| Component | Baseline | Ablation |
|-----------|----------|----------|
| ViT backbone | ViT-S/16 (384-d) | Same |
| Head MLP | 384→2048→2048→256 | Same |
| Head output | 256→**65536** | 256→**4096** |
| Weight norm last layer | Yes | Yes |
| Student params | ~43.8M | ~27.9M |

In [ ]:
# ══════════════════════════════════════════════
# ViT + DINO Head
# ══════════════════════════════════════════════

class DropPath(nn.Module):
    def __init__(self, drop_prob=0.):
        super().__init__()
        self.drop_prob = drop_prob

    def forward(self, x):
        if not self.training or self.drop_prob == 0.:
            return x
        keep = 1 - self.drop_prob
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        mask = torch.rand(shape, device=x.device) < keep
        return x / keep * mask


class Block(nn.Module):
    def __init__(self, dim, num_heads, mlp_ratio=4.0, drop_path=0.):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, num_heads, batch_first=True)
        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, int(dim * mlp_ratio)),
            nn.GELU(),
            nn.Linear(int(dim * mlp_ratio), dim),
        )

    def forward(self, x):
        y = self.norm1(x)
        y, _ = self.attn(y, y, y, need_weights=False)
        x = x + self.drop_path(y)
        x = x + self.drop_path(self.mlp(self.norm2(x)))
        return x


class ViTSmall(nn.Module):
    """ViT-Small/16 backbone. Supports variable input sizes via interpolated pos embed."""

    def __init__(self, img_size=192, patch_size=16, in_chans=1,
                 embed_dim=384, depth=12, num_heads=6,
                 mlp_ratio=4.0, drop_path_rate=0.1):
        super().__init__()
        self.patch_size = patch_size
        self.embed_dim = embed_dim
        self.num_patches = (img_size // patch_size) ** 2

        self.patch_embed = nn.Conv2d(in_chans, embed_dim,
                                     kernel_size=patch_size, stride=patch_size)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches + 1, embed_dim))

        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, depth)]
        self.blocks = nn.ModuleList([
            Block(embed_dim, num_heads, mlp_ratio, drop_path=dpr[i])
            for i in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim)
        self._init_weights()

    def _init_weights(self):
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        self.apply(self._init_module)

    @staticmethod
    def _init_module(m):
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.LayerNorm):
            nn.init.ones_(m.weight)
            nn.init.zeros_(m.bias)

    def interpolate_pos_embed(self, x, h, w):
        """Interpolate positional embeddings for different input sizes."""
        N = x.shape[1] - 1  # number of patch tokens
        if N == self.num_patches:
            return self.pos_embed
        cls_pe = self.pos_embed[:, :1, :]
        patch_pe = self.pos_embed[:, 1:, :]
        dim = patch_pe.shape[-1]
        h0 = int(self.num_patches ** 0.5)
        patch_pe = patch_pe.reshape(1, h0, h0, dim).permute(0, 3, 1, 2)
        patch_pe = F.interpolate(patch_pe, size=(h, w), mode='bicubic', align_corners=False)
        patch_pe = patch_pe.permute(0, 2, 3, 1).reshape(1, -1, dim)
        return torch.cat([cls_pe, patch_pe], dim=1)

    def forward(self, x):
        B, C, H, W = x.shape
        x = self.patch_embed(x).flatten(2).transpose(1, 2)
        h, w = H // self.patch_size, W // self.patch_size

        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)
        x = x + self.interpolate_pos_embed(x, h, w)

        for blk in self.blocks:
            x = blk(x)
        x = self.norm(x)
        return x[:, 0]  # CLS token → (B, embed_dim)


class DINOHead(nn.Module):
    """DINO projection head: MLP → bottleneck → L2-norm → weight-norm output."""

    def __init__(self, in_dim, hidden_dim=2048, bottleneck_dim=256,
                 out_dim=4096, nlayers=3):
        super().__init__()
        layers = []
        for i in range(nlayers - 1):
            d_in = in_dim if i == 0 else hidden_dim
            layers.extend([
                nn.Linear(d_in, hidden_dim),
                nn.GELU(),
            ])
        layers.append(nn.Linear(hidden_dim, bottleneck_dim))
        self.mlp = nn.Sequential(*layers)
        self.last_layer = nn.utils.weight_norm(nn.Linear(bottleneck_dim, out_dim, bias=False))
        self.last_layer.weight_g.data.fill_(1)

    def forward(self, x):
        x = self.mlp(x)
        x = F.normalize(x, dim=-1)
        return self.last_layer(x)


class DINOModel(nn.Module):
    """Wrapper: ViT backbone + DINO head."""
    def __init__(self, backbone, head):
        super().__init__()
        self.backbone = backbone
        self.head = head

    def forward(self, x):
        return self.head(self.backbone(x))

    def extract_features(self, x):
        return self.backbone(x)


def count_params(model):
    return sum(p.numel() for p in model.parameters())


# Architecture check
_bb = ViTSmall(IMG_SIZE, PATCH_SIZE, 1, ENC_DIM, ENC_DEPTH, ENC_HEADS, MLP_RATIO, DROP_PATH)
_hd = DINOHead(ENC_DIM, HEAD_HIDDEN, HEAD_BOTTLENECK, HEAD_OUT_DIM, HEAD_NLAYERS)
print(f"Backbone params: {count_params(_bb):,} ({count_params(_bb)/1e6:.1f}M)")
print(f"Head params: {count_params(_hd):,} ({count_params(_hd)/1e6:.1f}M)")
print(f"Total (student): {(count_params(_bb)+count_params(_hd)):,} ({(count_params(_bb)+count_params(_hd))/1e6:.1f}M)")
print(f"Total (student+teacher): {2*(count_params(_bb)+count_params(_hd)):,} ({2*(count_params(_bb)+count_params(_hd))/1e6:.1f}M)")
print(f"\n★ Baseline student was 43.8M — ablation is {(count_params(_bb)+count_params(_hd))/1e6:.1f}M")
print(f"  Head output reduced: 256×65536 = {256*65536:,} → 256×4096 = {256*4096:,} (−{256*(65536-4096):,} params)")

# Test forward with both crop sizes
with torch.no_grad():
    _m = DINOModel(_bb, _hd)
    g = _m(torch.randn(2, 1, 192, 192))
    l = _m(torch.randn(2, 1, 96, 96))
    print(f"Global output: {g.shape}, Local output: {l.shape}")
del _bb, _hd, _m, g, l

In [ ]:
# ══════════════════════════════════════════════
# DINO Loss
# ══════════════════════════════════════════════

class DINOLoss(nn.Module):
    """DINO loss with centering and sharpening.
    Teacher: global crops only, sharpened (low temp).
    Student: all crops, softened (high temp).
    Cross-entropy between teacher (target) and student (pred)."""

    def __init__(self, out_dim, n_global, n_local,
                 student_temp=0.1, teacher_temp=0.04,
                 center_momentum=0.9):
        super().__init__()
        self.student_temp = student_temp
        self.teacher_temp = teacher_temp
        self.center_momentum = center_momentum
        self.n_global = n_global
        self.n_crops = n_global + n_local
        self.register_buffer('center', torch.zeros(1, out_dim))

    def update_teacher_temp(self, temp):
        self.teacher_temp = temp

    @torch.no_grad()
    def update_center(self, teacher_output):
        """Update center (EMA of teacher mean)."""
        batch_center = teacher_output.mean(dim=0, keepdim=True)
        self.center = self.center * self.center_momentum + batch_center * (1 - self.center_momentum)

    def forward(self, student_output, teacher_output):
        """student_output: list of n_crops tensors. teacher_output: list of n_global tensors."""
        s_out = [s / self.student_temp for s in student_output]
        t_out = [(t - self.center) / self.teacher_temp for t in teacher_output]

        t_probs = [F.softmax(t, dim=-1).detach() for t in t_out]

        total_loss = 0
        n_loss_terms = 0
        for t_idx in range(self.n_global):
            for s_idx in range(self.n_crops):
                if s_idx == t_idx:  # skip same view
                    continue
                loss = -torch.sum(t_probs[t_idx] * F.log_softmax(s_out[s_idx], dim=-1), dim=-1)
                total_loss += loss.mean()
                n_loss_terms += 1

        total_loss /= n_loss_terms

        self.update_center(torch.cat(teacher_output))

        return total_loss

In [ ]:
# ══════════════════════════════════════════════
# Training Infrastructure
# ══════════════════════════════════════════════
TOTAL_TIME_BUDGET = 12 * 3600
SESSION_START = time.time()


def log_time_budget(phase=""):
    elapsed = time.time() - SESSION_START
    remaining = TOTAL_TIME_BUDGET - elapsed
    print(f"⏱ [{phase}] Elapsed: {elapsed/3600:.2f}h | Remaining: {remaining/3600:.2f}h "
          f"| Budget used: {100*elapsed/TOTAL_TIME_BUDGET:.1f}%")
    if remaining < 1800:
        print("  ⚠️ WARNING: Less than 30 min remaining!")
    return remaining


def log_gpu_memory(prefix=""):
    if device == "cuda":
        alloc = torch.cuda.memory_allocated() / 1e9
        peak = torch.cuda.max_memory_allocated() / 1e9
        total = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"  🖥 GPU [{prefix}]: {alloc:.2f}/{total:.1f} GB (peak: {peak:.2f} GB)")


def clear_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
    log_gpu_memory("after clear")
    log_time_budget("GPU cleared")


def cosine_scheduler(base_value, final_value, epochs, warmup_epochs=0, start_warmup_value=0):
    """Cosine schedule with optional warmup."""
    warmup = np.linspace(start_warmup_value, base_value, warmup_epochs)
    cos_epochs = epochs - warmup_epochs
    cosine = final_value + 0.5 * (base_value - final_value) * (
        1 + np.cos(np.pi * np.arange(cos_epochs) / cos_epochs))
    return np.concatenate([warmup, cosine])


@torch.no_grad()
def update_teacher(student, teacher, momentum):
    """EMA update of teacher from student."""
    for s_param, t_param in zip(student.parameters(), teacher.parameters()):
        t_param.data.mul_(momentum).add_((1 - momentum) * s_param.data)


@torch.no_grad()
def extract_all_features(model, loader):
    """Extract L2-normalized backbone features."""
    model.eval()
    feats, labels = [], []
    for imgs, y in tqdm(loader, desc="Extracting features"):
        imgs = imgs.to(device)
        with torch.amp.autocast("cuda", enabled=(device == "cuda")):
            f = model.extract_features(imgs).float()
        f = F.normalize(f, dim=1)
        feats.append(f.cpu().numpy())
        labels.append(y.numpy())
    return np.concatenate(feats), np.concatenate(labels)

In [ ]:
# ══════════════════════════════════════════════
# DINO Training Loop
# ══════════════════════════════════════════════
def train_dino(train_loader, val_loader_clean, save_dir, model_name="DINO"):
    os.makedirs(save_dir, exist_ok=True)

    remaining = log_time_budget(f"START {model_name}")
    if remaining < 3600:
        print("  ⚠️ CRITICAL: Less than 1h remaining!")

    print(f"\n{'='*60}")
    print(f"  🚀 Training: {model_name}")
    print(f"  📁 Save dir: {save_dir}")
    print(f"  📊 Train batches: {len(train_loader)}")
    print(f"  ⚙️  Config: {EPOCHS}ep, BS={BATCH_SIZE}, accum={GRAD_ACCUM}, "
          f"crops={N_GLOBAL_CROPS}g+{N_LOCAL_CROPS}l, LR={BASE_LR}")
    print(f"  ★ HEAD_OUT_DIM={HEAD_OUT_DIM} (ablation, was 65536)")
    print(f"{'='*60}\n")

    # Build student and teacher
    student_bb = ViTSmall(IMG_SIZE, PATCH_SIZE, 1, ENC_DIM, ENC_DEPTH, ENC_HEADS, MLP_RATIO, DROP_PATH)
    student_hd = DINOHead(ENC_DIM, HEAD_HIDDEN, HEAD_BOTTLENECK, HEAD_OUT_DIM, HEAD_NLAYERS)
    student = DINOModel(student_bb, student_hd).to(device)

    teacher_bb = ViTSmall(IMG_SIZE, PATCH_SIZE, 1, ENC_DIM, ENC_DEPTH, ENC_HEADS, MLP_RATIO, 0.0)
    teacher_hd = DINOHead(ENC_DIM, HEAD_HIDDEN, HEAD_BOTTLENECK, HEAD_OUT_DIM, HEAD_NLAYERS)
    teacher = DINOModel(teacher_bb, teacher_hd).to(device)
    teacher.load_state_dict(student.state_dict())
    for p in teacher.parameters():
        p.requires_grad = False

    print(f"  Student params: {count_params(student):,} ({count_params(student)/1e6:.1f}M)")
    log_gpu_memory("models loaded")

    # Loss
    dino_loss_fn = DINOLoss(
        HEAD_OUT_DIM, N_GLOBAL_CROPS, N_LOCAL_CROPS,
        STUDENT_TEMP, TEACHER_TEMP_START, CENTER_MOMENTUM
    ).to(device)

    # Optimizer (exclude teacher)
    param_groups = [
        {'params': [p for n, p in student.named_parameters() if 'bias' not in n and 'norm' not in n],
         'weight_decay': WD_START},
        {'params': [p for n, p in student.named_parameters() if 'bias' in n or 'norm' in n],
         'weight_decay': 0.0},
    ]
    optimizer = torch.optim.AdamW(param_groups, lr=BASE_LR, betas=(0.9, 0.999))
    scaler = torch.amp.GradScaler("cuda", enabled=(device == "cuda"))

    # Schedules
    lr_schedule = cosine_scheduler(BASE_LR, MIN_LR, EPOCHS, WARMUP_EPOCHS)
    wd_schedule = cosine_scheduler(WD_START, WD_END, EPOCHS)
    momentum_schedule = cosine_scheduler(TEACHER_MOMENTUM_START, TEACHER_MOMENTUM_END, EPOCHS)
    teacher_temp_schedule = np.concatenate([
        np.linspace(TEACHER_TEMP_START, TEACHER_TEMP_END, TEACHER_TEMP_WARMUP),
        np.full(EPOCHS - TEACHER_TEMP_WARMUP, TEACHER_TEMP_END)
    ])

    history = {"loss": [], "lr": [], "wd": [], "teacher_mom": [],
               "teacher_temp": [], "epoch_time": []}
    best_loss = float("inf")
    total_start = time.time()

    for epoch in range(EPOCHS):
        epoch_start = time.time()
        student.train()
        teacher.eval()

        # Update schedules
        lr_now = lr_schedule[epoch]
        wd_now = wd_schedule[epoch]
        mom_now = momentum_schedule[epoch]
        temp_now = teacher_temp_schedule[epoch]
        dino_loss_fn.update_teacher_temp(temp_now)

        for i, pg in enumerate(optimizer.param_groups):
            pg['lr'] = lr_now
            if i == 0:
                pg['weight_decay'] = wd_now

        total_loss = 0.0
        num_batches = len(train_loader)
        log_interval = max(1, num_batches // 5)
        optimizer.zero_grad()

        for step, (crops, _) in enumerate(train_loader):
            # Forward all crops through student
            student_out = []
            for c in crops:
                c = c.to(device)
                with torch.amp.autocast("cuda", enabled=(device == "cuda")):
                    student_out.append(student(c))

            # Forward global crops through teacher
            with torch.no_grad():
                teacher_out = []
                for c in crops[:N_GLOBAL_CROPS]:
                    c = c.to(device)
                    with torch.amp.autocast("cuda", enabled=(device == "cuda")):
                        teacher_out.append(teacher(c))

            with torch.amp.autocast("cuda", enabled=(device == "cuda")):
                loss = dino_loss_fn(student_out, teacher_out) / GRAD_ACCUM

            scaler.scale(loss).backward()

            if (step + 1) % GRAD_ACCUM == 0 or (step + 1) == num_batches:
                scaler.unscale_(optimizer)
                # Cancel gradients for last layer during first epoch (DINO warmup trick)
                if epoch < 1:
                    for p in student.head.last_layer.parameters():
                        p.grad = None
                torch.nn.utils.clip_grad_norm_(student.parameters(), MAX_GRAD_NORM)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()

                # EMA update teacher
                with torch.no_grad():
                    update_teacher(student, teacher, mom_now)

            total_loss += loss.item() * GRAD_ACCUM

            if (step + 1) % log_interval == 0:
                avg = total_loss / (step + 1)
                elapsed = time.time() - epoch_start
                rate = (step + 1) / elapsed
                eta = (num_batches - step - 1) / rate
                print(f"    batch {step+1}/{num_batches} | loss={avg:.4f} | "
                      f"{rate:.1f} batch/s | ETA: {eta:.0f}s", end="\r")

        print(" " * 80, end="\r")
        epoch_loss = total_loss / num_batches
        epoch_time = time.time() - epoch_start

        history["loss"].append(epoch_loss)
        history["lr"].append(lr_now)
        history["wd"].append(wd_now)
        history["teacher_mom"].append(mom_now)
        history["teacher_temp"].append(temp_now)
        history["epoch_time"].append(epoch_time)

        elapsed_model = time.time() - total_start
        elapsed_session = time.time() - SESSION_START
        remaining_session = TOTAL_TIME_BUDGET - elapsed_session

        print(f"  [{model_name}] Epoch {epoch+1:2d}/{EPOCHS} │ "
              f"loss={epoch_loss:.4f} │ lr={lr_now:.6f} wd={wd_now:.4f} │ "
              f"mom={mom_now:.4f} t_temp={temp_now:.4f} │ "
              f"{epoch_time:.0f}s/ep │ budget left: {remaining_session/3600:.2f}h")

        if epoch_loss < best_loss:
            best_loss = epoch_loss
            torch.save({"epoch": epoch + 1,
                        "student": student.state_dict(),
                        "teacher": teacher.state_dict(),
                        "loss": epoch_loss},
                       os.path.join(save_dir, "best.pt"))
            print(f"       ↑ New best! (loss={epoch_loss:.4f})")

        if (epoch + 1) % 5 == 0:
            torch.save({"epoch": epoch + 1,
                        "student": student.state_dict(),
                        "teacher": teacher.state_dict()},
                       os.path.join(save_dir, f"epoch_{epoch+1}.pt"))
            print(f"       💾 Checkpoint: epoch_{epoch+1}.pt")
            log_gpu_memory(f"epoch {epoch+1}")

        if remaining_session < 1800 and epoch < EPOCHS - 1:
            print(f"  ⚠️ TIME: {remaining_session/60:.0f} min left! Stopping at epoch {epoch+1}.")
            break

    total_time = time.time() - total_start
    torch.save({"epoch": epoch + 1,
                "student": student.state_dict(),
                "teacher": teacher.state_dict(),
                "history": history},
               os.path.join(save_dir, "final.pt"))

    print(f"\n  ✅ {model_name} complete in {total_time/60:.1f} min ({total_time/3600:.2f}h)")
    print(f"     Best loss: {best_loss:.4f}")
    print(f"     Avg epoch time: {np.mean(history['epoch_time']):.1f}s")
    log_time_budget(f"END {model_name}")

    # Load best teacher (used for feature extraction)
    best = torch.load(os.path.join(save_dir, "best.pt"), map_location=device, weights_only=False)
    teacher.load_state_dict(best["teacher"])

    return teacher, history

In [ ]:
# ══════════════════════════════════════════════
# Full Analysis Pipeline
# ══════════════════════════════════════════════
def run_full_analysis(model, train_loader, val_loader,
                      val_dataset, history, save_dir, model_name):
    analysis_start = time.time()
    log_time_budget(f"ANALYSIS START {model_name}")

    print(f"\n{'='*60}")
    print(f"  📈 Analysis: {model_name}")
    print(f"{'='*60}")

    # ── 1. Training Curves ──
    print("\n[1/10] Training curves...")
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle(f"{model_name} — Training Curves & Schedules", fontsize=14, y=1.02)
    axes[0][0].plot(history['loss'], linewidth=2, color='steelblue')
    axes[0][0].set_xlabel('Epoch'); axes[0][0].set_ylabel('DINO Loss')
    axes[0][0].set_title('Loss'); axes[0][0].grid(True, alpha=0.3)
    # Mark ln(4096) collapse threshold
    axes[0][0].axhline(y=np.log(HEAD_OUT_DIM), color='red', linestyle='--', alpha=0.5,
                       label=f'ln({HEAD_OUT_DIM})={np.log(HEAD_OUT_DIM):.2f}')
    axes[0][0].legend(fontsize=8)
    axes[0][1].plot(history['lr'], linewidth=2, color='green')
    axes[0][1].set_xlabel('Epoch'); axes[0][1].set_ylabel('LR')
    axes[0][1].set_title('Learning Rate'); axes[0][1].grid(True, alpha=0.3)
    axes[0][2].plot(history['wd'], linewidth=2, color='orange')
    axes[0][2].set_xlabel('Epoch'); axes[0][2].set_ylabel('WD')
    axes[0][2].set_title('Weight Decay'); axes[0][2].grid(True, alpha=0.3)
    axes[1][0].plot(history['teacher_mom'], linewidth=2, color='purple')
    axes[1][0].set_xlabel('Epoch'); axes[1][0].set_ylabel('Momentum')
    axes[1][0].set_title('Teacher Momentum'); axes[1][0].grid(True, alpha=0.3)
    axes[1][1].plot(history['teacher_temp'], linewidth=2, color='red')
    axes[1][1].set_xlabel('Epoch'); axes[1][1].set_ylabel('Temperature')
    axes[1][1].set_title('Teacher Temperature'); axes[1][1].grid(True, alpha=0.3)
    axes[1][2].plot(history['epoch_time'], linewidth=2, color='brown')
    axes[1][2].set_xlabel('Epoch'); axes[1][2].set_ylabel('Seconds')
    axes[1][2].set_title('Epoch Time'); axes[1][2].grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(os.path.join(save_dir, 'training_curves.png'), dpi=150, bbox_inches='tight')
    plt.show()

    # ── 2. Feature Extraction ──
    print("[2/10] Extracting teacher features...")
    feat_start = time.time()
    train_feats, train_labels = extract_all_features(model, train_loader)
    val_feats, val_labels = extract_all_features(model, val_loader)
    print(f"  Train: {train_feats.shape}, Val: {val_feats.shape} ({time.time()-feat_start:.1f}s)")

    # ── 3. Collapse Check ──
    print("[3/10] Collapse check...")
    n_check = min(2000, len(train_feats))
    for name, feats in [('Train', train_feats[:n_check]), ('Val', val_feats[:n_check])]:
        std = np.std(feats, axis=0).mean()
        sim = np.dot(feats, feats.T)
        off = sim[np.triu_indices_from(sim, k=1)]
        s_std = '✅' if std > 0.01 else '⚠️ COLLAPSE'
        s_sim = '✅' if off.mean() < 0.9 else '⚠️ HIGH'
        print(f"  {name}: STD={std:.4f} {s_std} | Mean cos sim={off.mean():.4f} {s_sim}")

    # ── 4. kNN Anomaly Scores ──
    print("[4/10] kNN anomaly scores...")
    nn_model = NearestNeighbors(n_neighbors=K_NEIGHBORS, metric='cosine')
    nn_model.fit(train_feats)
    train_knn = nn_model.kneighbors(train_feats)[0].mean(axis=1)
    val_knn = nn_model.kneighbors(val_feats)[0].mean(axis=1)
    print(f"  Train kNN — mean: {train_knn.mean():.4f}, std: {train_knn.std():.4f}")
    print(f"  Val kNN   — mean: {val_knn.mean():.4f}, std: {val_knn.std():.4f}")

    # ── 5. Score Distributions ──
    print("[5/10] Score distributions...")
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f"{model_name} — kNN Anomaly Score Distributions", fontsize=14, y=1.02)
    axes[0].hist(train_knn, bins=60, alpha=0.7, color='steelblue', edgecolor='white')
    axes[0].set_title('Train — kNN Distance'); axes[0].set_xlabel('Cosine Dist')
    axes[1].hist(val_knn, bins=60, alpha=0.7, color='coral', edgecolor='white')
    axes[1].set_title('Val — kNN Distance'); axes[1].set_xlabel('Cosine Dist')
    fig.tight_layout()
    fig.savefig(os.path.join(save_dir, 'anomaly_scores.png'), dpi=150, bbox_inches='tight')
    plt.show()

    # ── 6. Threshold Analysis ──
    print("[6/10] Threshold analysis...")
    print("  kNN-based:")
    for pct in [90, 95, 97.5, 99]:
        thresh = np.percentile(train_knn, pct)
        flagged = (val_knn > thresh).sum()
        print(f"    p{pct}: thresh={thresh:.4f} → {flagged}/{len(val_knn)} val flagged "
              f"({100*flagged/len(val_knn):.1f}%)")

    # ── 7. t-SNE ──
    print("[7/10] t-SNE...")
    tsne_start = time.time()
    N_TSNE = min(3000, len(val_feats))
    tsne = TSNE(n_components=2, perplexity=30, random_state=SEED, max_iter=1000)
    emb = tsne.fit_transform(val_feats[:N_TSNE])
    labels_sub = val_labels[:N_TSNE]
    knn_sub = val_knn[:N_TSNE]
    print(f"  t-SNE done ({time.time()-tsne_start:.1f}s)")

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    fig.suptitle(f"{model_name} — t-SNE Embeddings", fontsize=14, y=1.02)
    axes[0].scatter(emb[:, 0], emb[:, 1], s=3, alpha=0.5, c='steelblue')
    axes[0].set_title('Unsupervised'); axes[0].set_xticks([]); axes[0].set_yticks([])
    sc1 = axes[1].scatter(emb[:, 0], emb[:, 1], s=3, alpha=0.5, c=labels_sub, cmap='coolwarm')
    axes[1].set_title('By Label'); axes[1].set_xticks([]); axes[1].set_yticks([])
    plt.colorbar(sc1, ax=axes[1], shrink=0.8)
    sc2 = axes[2].scatter(emb[:, 0], emb[:, 1], s=3, alpha=0.5, c=knn_sub, cmap='YlOrRd')
    axes[2].set_title('kNN Score'); axes[2].set_xticks([]); axes[2].set_yticks([])
    plt.colorbar(sc2, ax=axes[2], shrink=0.8)
    fig.tight_layout()
    fig.savefig(os.path.join(save_dir, 'tsne.png'), dpi=150, bbox_inches='tight')
    plt.show()

    # ── 8. Cosine Similarity Distribution ──
    print("[8/10] Cosine similarity distribution...")
    subset = torch.tensor(val_feats[:2000])
    sim_matrix = torch.mm(subset, subset.T).numpy()
    upper_tri = sim_matrix[np.triu_indices_from(sim_matrix, k=1)]
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(upper_tri, bins=100, color='mediumpurple', edgecolor='white', alpha=0.8)
    ax.axvline(upper_tri.mean(), color='red', ls='--', label=f'Mean: {upper_tri.mean():.3f}')
    ax.set_title(f"{model_name} — Pairwise Cosine Similarity")
    ax.set_xlabel('Cosine Similarity'); ax.legend()
    fig.tight_layout()
    fig.savefig(os.path.join(save_dir, 'similarity_dist.png'), dpi=150, bbox_inches='tight')
    plt.show()

    # ── 9. NN Retrieval ──
    print("[9/10] Nearest neighbor retrieval...")
    nn_vis = NearestNeighbors(n_neighbors=6, metric='cosine').fit(val_feats)
    indices = random.sample(range(len(val_feats)), 3)
    fig, axes = plt.subplots(3, 6, figsize=(15, 7.5))
    fig.suptitle(f"{model_name} — NN Retrieval", fontsize=14, y=1.02)
    for row, q_idx in enumerate(indices):
        _, nn_idxs = nn_vis.kneighbors([val_feats[q_idx]])
        for col, idx in enumerate(nn_idxs[0]):
            img = np.load(val_dataset.paths[idx])
            axes[row][col].imshow(img, cmap='gray', vmin=0, vmax=1)
            axes[row][col].axis('off')
            axes[row][col].set_title('Query' if col == 0 else f'NN-{col}', fontsize=9)
    fig.tight_layout()
    fig.savefig(os.path.join(save_dir, 'nn_retrieval.png'), dpi=150, bbox_inches='tight')
    plt.show()

    # ── 10. Top Anomalies ──
    print("[10/10] Top anomalies...")
    fig, axes = plt.subplots(2, 5, figsize=(15, 6))
    fig.suptitle(f"{model_name} — Top-10 Anomalies (by kNN distance)", fontsize=14)
    top_idx = np.argsort(val_knn)[-10:][::-1]
    for i, idx in enumerate(top_idx):
        ax = axes[i // 5][i % 5]
        img = np.load(val_dataset.paths[idx])
        ax.imshow(img, cmap='gray', vmin=0, vmax=1)
        ax.set_title(f'kNN={val_knn[idx]:.4f}', fontsize=8)
        ax.axis('off')
    fig.tight_layout()
    fig.savefig(os.path.join(save_dir, 'top_anomalies.png'), dpi=150, bbox_inches='tight')
    plt.show()

    # ── Save ──
    np.savez(
        os.path.join(save_dir, 'scoring.npz'),
        train_knn=train_knn, val_knn=val_knn,
        train_feats=train_feats, val_feats=val_feats,
        train_labels=train_labels, val_labels=val_labels,
        threshold_knn_p95=np.percentile(train_knn, 95),
    )

    print(f"\n  ✅ Analysis complete in {(time.time()-analysis_start)/60:.1f} min")
    print(f"  📁 Files: {os.listdir(save_dir)}")
    log_time_budget(f"ANALYSIS END {model_name}")

    return train_feats, val_feats, train_knn, val_knn

In [ ]:
# ══════════════════════════════════════════════
# Discover dataset paths
# ══════════════════════════════════════════════
KNEE_ROOT = "/kaggle/input/preprocessed-fastmri-knee"
BRAIN_ROOT = "/kaggle/input/preprocessed-ixi-brain"

def show_tree(path, prefix="", max_depth=3, depth=0):
    if depth >= max_depth:
        return
    try:
        entries = sorted(os.listdir(path))
    except (PermissionError, FileNotFoundError) as e:
        print(f"{prefix}⚠️ {e}")
        return
    dirs = [e for e in entries if os.path.isdir(os.path.join(path, e))]
    files = [e for e in entries if os.path.isfile(os.path.join(path, e))]
    if files:
        npy = [f for f in files if f.endswith('.npy')]
        if npy:
            print(f"{prefix}📄 {len(npy)} .npy files (e.g., {npy[0]})")
        for f in [x for x in files if not x.endswith('.npy')][:3]:
            print(f"{prefix}📄 {f}")
    for d in dirs:
        print(f"{prefix}📁 {d}/")
        show_tree(os.path.join(path, d), prefix + "    ", max_depth, depth + 1)

print("=" * 50)
print("KNEE Dataset:")
print("=" * 50)
show_tree(KNEE_ROOT)

print(f"\n{'='*50}")
print("BRAIN Dataset:")
print("=" * 50)
show_tree(BRAIN_ROOT)

In [ ]:
# ══════════════════════════════════════════════
# Configure Paths + Balanced Brain Sampling
# ══════════════════════════════════════════════
KNEE_TRAIN = os.path.join(KNEE_ROOT, "train")
KNEE_VAL   = os.path.join(KNEE_ROOT, "val")
BRAIN_TRAIN = os.path.join(BRAIN_ROOT, "train")
BRAIN_VAL   = os.path.join(BRAIN_ROOT, "val")

print("Knee (all):")
knee_train_paths, knee_train_labels = get_balanced_paths(KNEE_TRAIN)
knee_val_paths, knee_val_labels = get_balanced_paths(KNEE_VAL)
print(f"  Train: {len(knee_train_paths)} | Val: {len(knee_val_paths)}")

print(f"\nBrain (balanced to match knee = {len(knee_train_paths)}):")
print("  Train:")
brain_train_paths, brain_train_labels = get_balanced_paths(
    BRAIN_TRAIN, target_total=len(knee_train_paths))
print("  Val:")
brain_val_paths, brain_val_labels = get_balanced_paths(
    BRAIN_VAL, target_total=len(knee_val_paths))
print(f"  Train: {len(brain_train_paths)} | Val: {len(brain_val_paths)}")

knee_label_map = {l: i for i, l in enumerate(sorted(set(knee_train_labels)))}
brain_label_map = {l: i for i, l in enumerate(sorted(set(brain_train_labels)))}
all_labels = sorted(set(knee_train_labels + brain_train_labels))
combined_label_map = {l: i for i, l in enumerate(all_labels)}

print(f"\nLabel maps:")
print(f"  Knee: {knee_label_map}")
print(f"  Brain: {brain_label_map}")
print(f"  Combined: {combined_label_map}")

---
# Part 1: Knee Only (FastMRI)

In [ ]:
# ══════════════════════════════════════════════
# PART 1: KNEE — Data Loaders
# ══════════════════════════════════════════════
print("Loading Knee datasets...")
dino_aug = DINOMultiCropTransform(IMG_SIZE, LOCAL_CROP_SIZE, N_GLOBAL_CROPS, N_LOCAL_CROPS)

knee_train_ds = MRIDataset(knee_train_paths, knee_train_labels, knee_label_map,
                           transform=dino_aug, name="Knee-Train")
knee_train_loader = DataLoader(knee_train_ds, batch_size=BATCH_SIZE, shuffle=True,
                               num_workers=NUM_WORKERS, pin_memory=True,
                               drop_last=True, collate_fn=dino_collate)

# Clean loaders (single image, no multi-crop)
knee_train_clean = MRIDataset(knee_train_paths, knee_train_labels, knee_label_map,
                              name="Knee-Train-Clean")
knee_val_clean = MRIDataset(knee_val_paths, knee_val_labels, knee_label_map,
                            name="Knee-Val-Clean")

knee_train_clean_loader = DataLoader(knee_train_clean, batch_size=BATCH_SIZE * 2, shuffle=False,
                                     num_workers=NUM_WORKERS, pin_memory=True)
knee_val_clean_loader = DataLoader(knee_val_clean, batch_size=BATCH_SIZE * 2, shuffle=False,
                                   num_workers=NUM_WORKERS, pin_memory=True)

print(f"\nTrain batches: {len(knee_train_loader)}")
print(f"Val clean: {len(knee_val_clean)} slices")

In [ ]:
# ══════════════════════════════════════════════
# PART 1: KNEE — Train DINO (Ablation: HEAD_OUT_DIM=4096)
# ══════════════════════════════════════════════
KNEE_SAVE = "/kaggle/working/checkpoints/dino_ablation_knee"
knee_teacher, knee_history = train_dino(
    knee_train_loader, knee_val_clean_loader, KNEE_SAVE, model_name="DINO-Abl-Knee")

In [ ]:
# ══════════════════════════════════════════════
# PART 1: KNEE — Full Analysis
# ══════════════════════════════════════════════
knee_results = run_full_analysis(
    knee_teacher, knee_train_clean_loader, knee_val_clean_loader,
    knee_val_clean, knee_history, KNEE_SAVE, "DINO-Abl-Knee")

In [ ]:
del knee_teacher
del knee_train_loader, knee_train_clean_loader, knee_val_clean_loader
clear_gpu()

---
# Part 2: Brain Only (IXI — Balanced T1 + T2)

In [ ]:
# ══════════════════════════════════════════════
# PART 2: BRAIN — Data Loaders (balanced)
# ══════════════════════════════════════════════
print("Loading Brain datasets (balanced)...")
dino_aug = DINOMultiCropTransform(IMG_SIZE, LOCAL_CROP_SIZE, N_GLOBAL_CROPS, N_LOCAL_CROPS)

brain_train_ds = MRIDataset(brain_train_paths, brain_train_labels, brain_label_map,
                            transform=dino_aug, name="Brain-Train")
brain_train_loader = DataLoader(brain_train_ds, batch_size=BATCH_SIZE, shuffle=True,
                                num_workers=NUM_WORKERS, pin_memory=True,
                                drop_last=True, collate_fn=dino_collate)

brain_train_clean = MRIDataset(brain_train_paths, brain_train_labels, brain_label_map,
                               name="Brain-Train-Clean")
brain_val_clean = MRIDataset(brain_val_paths, brain_val_labels, brain_label_map,
                             name="Brain-Val-Clean")

brain_train_clean_loader = DataLoader(brain_train_clean, batch_size=BATCH_SIZE * 2, shuffle=False,
                                      num_workers=NUM_WORKERS, pin_memory=True)
brain_val_clean_loader = DataLoader(brain_val_clean, batch_size=BATCH_SIZE * 2, shuffle=False,
                                    num_workers=NUM_WORKERS, pin_memory=True)

print(f"\nTrain batches: {len(brain_train_loader)}")
print(f"Brain train: {len(brain_train_ds)} | Val clean: {len(brain_val_clean)}")

In [ ]:
# ══════════════════════════════════════════════
# PART 2: BRAIN — Train DINO (Ablation: HEAD_OUT_DIM=4096)
# ══════════════════════════════════════════════
BRAIN_SAVE = "/kaggle/working/checkpoints/dino_ablation_brain"
brain_teacher, brain_history = train_dino(
    brain_train_loader, brain_val_clean_loader, BRAIN_SAVE, model_name="DINO-Abl-Brain")

In [ ]:
# ══════════════════════════════════════════════
# PART 2: BRAIN — Full Analysis
# ══════════════════════════════════════════════
brain_results = run_full_analysis(
    brain_teacher, brain_train_clean_loader, brain_val_clean_loader,
    brain_val_clean, brain_history, BRAIN_SAVE, "DINO-Abl-Brain")

In [ ]:
del brain_teacher
del brain_train_loader, brain_train_clean_loader, brain_val_clean_loader
clear_gpu()

---
# Part 3: Combined (Knee + Balanced Brain)

In [ ]:
# ══════════════════════════════════════════════
# PART 3: COMBINED — Data Loaders
# ══════════════════════════════════════════════
print("Loading Combined (Knee + balanced Brain) datasets...")
dino_aug = DINOMultiCropTransform(IMG_SIZE, LOCAL_CROP_SIZE, N_GLOBAL_CROPS, N_LOCAL_CROPS)

comb_train_paths = knee_train_paths + brain_train_paths
comb_train_labels = knee_train_labels + brain_train_labels
comb_val_paths = knee_val_paths + brain_val_paths
comb_val_labels = knee_val_labels + brain_val_labels

comb_train_ds = MRIDataset(comb_train_paths, comb_train_labels, combined_label_map,
                           transform=dino_aug, name="Combined-Train")
comb_train_loader = DataLoader(comb_train_ds, batch_size=BATCH_SIZE, shuffle=True,
                               num_workers=NUM_WORKERS, pin_memory=True,
                               drop_last=True, collate_fn=dino_collate)

comb_train_clean = MRIDataset(comb_train_paths, comb_train_labels, combined_label_map,
                              name="Combined-Train-Clean")
comb_val_clean = MRIDataset(comb_val_paths, comb_val_labels, combined_label_map,
                            name="Combined-Val-Clean")

comb_train_clean_loader = DataLoader(comb_train_clean, batch_size=BATCH_SIZE * 2, shuffle=False,
                                     num_workers=NUM_WORKERS, pin_memory=True)
comb_val_clean_loader = DataLoader(comb_val_clean, batch_size=BATCH_SIZE * 2, shuffle=False,
                                   num_workers=NUM_WORKERS, pin_memory=True)

print(f"\nCombined train: {len(comb_train_ds)} slices, {len(comb_train_loader)} batches")
print(f"Combined val: {len(comb_val_clean)} slices")

In [ ]:
# ══════════════════════════════════════════════
# PART 3: COMBINED — Train DINO (Ablation: HEAD_OUT_DIM=4096)
# ══════════════════════════════════════════════
COMBINED_SAVE = "/kaggle/working/checkpoints/dino_ablation_combined"
comb_teacher, comb_history = train_dino(
    comb_train_loader, comb_val_clean_loader, COMBINED_SAVE, model_name="DINO-Abl-Combined")

In [ ]:
# ══════════════════════════════════════════════
# PART 3: COMBINED — Full Analysis
# ══════════════════════════════════════════════
comb_results = run_full_analysis(
    comb_teacher, comb_train_clean_loader, comb_val_clean_loader,
    comb_val_clean, comb_history, COMBINED_SAVE, "DINO-Abl-Combined")

In [ ]:
del comb_teacher
del comb_train_loader, comb_train_clean_loader, comb_val_clean_loader
clear_gpu()

---
# Summary & Comparison

In [ ]:
# ══════════════════════════════════════════════
# Cross-Model Comparison
# ══════════════════════════════════════════════
print("\n" + "=" * 80)
print("  DINO ABLATION (HEAD_OUT_DIM=4096) — MODEL COMPARISON SUMMARY")
print("=" * 80)

results = {}
for name, save_dir in [("Knee", KNEE_SAVE), ("Brain", BRAIN_SAVE), ("Combined", COMBINED_SAVE)]:
    data = np.load(os.path.join(save_dir, "scoring.npz"), allow_pickle=True)
    results[name] = {
        "train_knn": data["train_knn"],
        "val_knn": data["val_knn"],
        "val_feats": data["val_feats"],
        "thresh_knn": float(data["threshold_knn_p95"]),
    }

# kNN scores
print(f"\n{'Model':<12} {'Train μ':<10} {'Val μ':<10} {'Val σ':<10} {'p95 Thresh':<12} {'Val>p95':}")
print("-" * 70)
for name, r in results.items():
    tr, vr = r["train_knn"], r["val_knn"]
    flagged = (vr > r["thresh_knn"]).sum()
    print(f"{name:<12} {tr.mean():<10.4f} {vr.mean():<10.4f} {vr.std():<10.4f} "
          f"{r['thresh_knn']:<12.4f} {flagged}/{len(vr)} ({100*flagged/len(vr):.1f}%)")

# Feature diversity
print(f"\n{'Model':<12} {'Feature STD':<14} {'Mean Cos Sim':<14} {'Status'}")
print("-" * 54)
for name, r in results.items():
    feats = r["val_feats"][:2000]
    std = np.std(feats, axis=0).mean()
    sim = np.dot(feats, feats.T)
    off = sim[np.triu_indices_from(sim, k=1)].mean()
    status = "✅ Good" if off < 0.9 else "⚠️ High sim"
    print(f"{name:<12} {std:<14.4f} {off:<14.4f} {status}")

In [ ]:
# ══════════════════════════════════════════════
# Comparative Plots
# ══════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("DINO Ablation (OUT_DIM=4096) — Model Comparison", fontsize=14, y=1.02)

colors = {"Knee": "tab:blue", "Brain": "tab:orange", "Combined": "tab:green"}

# Load histories
histories = {}
for name, save_dir in [("Knee", KNEE_SAVE), ("Brain", BRAIN_SAVE), ("Combined", COMBINED_SAVE)]:
    ckpt = torch.load(os.path.join(save_dir, "final.pt"), map_location="cpu", weights_only=False)
    histories[name] = ckpt["history"]

for name, h in histories.items():
    axes[0].plot(h['loss'], label=name, linewidth=2, color=colors[name])
# Mark collapse thresholds
axes[0].axhline(y=np.log(4096), color='red', linestyle='--', alpha=0.4, label=f'ln(4096)={np.log(4096):.2f}')
axes[0].axhline(y=np.log(65536), color='gray', linestyle=':', alpha=0.4, label=f'ln(65536)={np.log(65536):.2f} (baseline)')
axes[0].set_title('DINO Loss'); axes[0].set_xlabel('Epoch')
axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)

for name, r in results.items():
    axes[1].hist(r['val_knn'], bins=50, alpha=0.5, label=name, color=colors[name])
axes[1].set_title('Val kNN Scores'); axes[1].set_xlabel('Cosine Dist')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

for name, h in histories.items():
    axes[2].bar(name, np.mean(h['epoch_time']), color=colors[name])
axes[2].set_title('Avg Epoch Time (s)'); axes[2].grid(True, alpha=0.3)

fig.tight_layout()
fig.savefig('dino_ablation_comparison_plots.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ══════════════════════════════════════════════
# Baseline vs Ablation Comparison Table
# ══════════════════════════════════════════════
print("\n" + "=" * 90)
print("  BASELINE (65536) vs ABLATION (4096) — Side-by-Side")
print("=" * 90)
print(f"\n{'Model':<12} {'Baseline Cos Sim':<20} {'Ablation Cos Sim':<20} {'Baseline Status':<18} {'Ablation Status'}")
print("-" * 90)

# Baseline results (hardcoded from previous run)
baseline = {
    "Knee":     {"cos_sim": 0.0112, "std": 0.0427, "status": "Borderline"},
    "Brain":    {"cos_sim": 0.9973, "std": 0.0022, "status": "⚠️ COLLAPSED"},
    "Combined": {"cos_sim": 0.2880, "std": 0.0372, "status": "✅ Good"},
}

for name, r in results.items():
    feats = r["val_feats"][:2000]
    std = np.std(feats, axis=0).mean()
    sim = np.dot(feats, feats.T)
    off = sim[np.triu_indices_from(sim, k=1)].mean()
    abl_status = "✅ Good" if off < 0.9 else "⚠️ COLLAPSED"
    b = baseline[name]
    print(f"{name:<12} {b['cos_sim']:<20.4f} {off:<20.4f} {b['status']:<18} {abl_status}")

print(f"\n{'='*90}")
print("Ablation conclusion:")
print("  If all models show ✅ Good → reducing HEAD_OUT_DIM prevents collapse.")
print("  The output dimensionality relative to dataset size is the key factor.")

In [ ]:
# ══════════════════════════════════════════════
# Zip & Download All Checkpoints
# ══════════════════════════════════════════════
import shutil
from IPython.display import FileLink, display

for name, save_dir in [("knee", KNEE_SAVE), ("brain", BRAIN_SAVE), ("combined", COMBINED_SAVE)]:
    zip_path = shutil.make_archive(f"dino_ablation_{name}_checkpoints", "zip",
                                   root_dir=".", base_dir=save_dir)
    size_mb = os.path.getsize(zip_path) / 1e6
    print(f"  {name}: {zip_path} ({size_mb:.1f} MB)")
    display(FileLink(zip_path))

all_zip = shutil.make_archive("dino_ablation_all_checkpoints", "zip",
                              root_dir=".", base_dir="/kaggle/working/checkpoints")
print(f"\n  ALL: {all_zip} ({os.path.getsize(all_zip)/1e6:.1f} MB)")
display(FileLink(all_zip))

log_time_budget("SESSION COMPLETE")